# Replication: Quality Factor
### optlab-research Â· Week 4 Validation

> **Before running:** Kernel â†’ Restart Kernel (clears lingering DuckDB state and
> recursion-guard flags). Then **Run All**.

| | |
|---|---|
| **Primary signal** | `gross_profitability` â€” (Revenue âˆ’ COGS) / Total Assets |
| **Secondary signal** | `roe` â€” Net Income / Book Equity |
| **Sort direction** | Standard: Q5 (high quality) long, Q1 (low quality) short |
| **Universe** | `russell1000` Â· 2019â€“2023 validation window |

**Literature:** Novy-Marx (2013) *JFE* 108(1); Asness, Frazzini & Pedersen (2019) *RAS*.


In [ ]:
from __future__ import annotations
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

import datetime as _dt, time
import duckdb, numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt, matplotlib.ticker as mtick
from pathlib import Path
from IPython.display import display

from optlab_research.backtest.engine import Backtest, BacktestConfig
from optlab_research.signals.compute import compute_signal

# â”€â”€ Connection (_CON_KEEPER at module level â†’ GC never closes it) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
try:
    from optlab_research import open_connection as _get_con
except (ImportError, AttributeError):
    from optlab.db import connect as _get_con

_CON_KEEPER = _get_con()
if isinstance(_CON_KEEPER, duckdb.DuckDBPyConnection):
    con = _CON_KEEPER
elif hasattr(_CON_KEEPER, '__enter__'):
    con = _CON_KEEPER.__enter__()
else:
    for _a in ('_con','con','connection','_connection','db','_db'):
        _c = getattr(_CON_KEEPER, _a, None)
        if isinstance(_c, duckdb.DuckDBPyConnection):
            con = _c; break
    else:
        raise RuntimeError(f"No DuckDB connection in {type(_CON_KEEPER)}")
print(f"Connection: {type(con).__name__}")


## Environment patches (run once per session)

Two issues in optlab that need fixing before any signal or backtest can run:

**Patch 1 â€” `optcrsp_link` stub**
`universe.py` has a `LEFT JOIN optcrsp_link` hardcoded even when `attach_secid=False`.
That table only exists with full OptionMetrics access. Fix: create an empty stub view
with the expected columns. The LEFT JOIN returns NULL secid for every row â€” which is
exactly what `attach_secid=False` should produce.

**Patch 2 â€” inject missing funda columns**
`get_universe_as_of(attach_funda=True)` does not include `cogs`. The patch wraps the
function, checks for missing columns after the normal call, and left-joins them in
from `comp_funda` with the same 90-day PIT constraint.

**Recursion guard:** the patch saves the true original on the module object
(`_univ_mod._true_orig`) so re-running this cell is safe â€” it always wraps the
original, never the already-patched version.


In [ ]:
import optlab.universe as _univ_mod

# â”€â”€ Patch 1: optcrsp_link stub â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
_need_stub = False
try:
    con.execute("SELECT secid FROM optcrsp_link LIMIT 0")
    print("optcrsp_link: already exists")
except Exception:
    _need_stub = True

if _need_stub:
    con.execute("""
        CREATE OR REPLACE VIEW optcrsp_link AS
        SELECT CAST(NULL AS INTEGER) AS secid,
               CAST(NULL AS INTEGER) AS permno,
               CAST(NULL AS DATE)    AS sdate,
               CAST(NULL AS DATE)    AS edate
        WHERE 1 = 0
    """)
    print("optcrsp_link: empty stub created")

# â”€â”€ Patch 2: funda column injection â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Recursion guard: store the TRUE original once on the module object.
# If this cell is re-run, _true_orig still points to the real function,
# not to the already-patched wrapper.
if not hasattr(_univ_mod, '_true_orig'):
    _univ_mod._true_orig = _univ_mod.get_universe_as_of
    print("Funda patch: saved true original")
else:
    print("Funda patch: true original already saved (cell re-run safe)")

_FUNDA_PATCH = ["at", "ceq", "ni", "oancf", "revt", "cogs", "sich"]

def _patched_get_universe(
    con, date,
    shrcds=(10, 11), exchcds=(1, 2, 3),
    min_price=1.0, min_mcap=None,
    attach_gvkey=False, attach_secid=False,
    attach_funda=False, require_options=False,
):
    # Always call the TRUE original â€” never self
    univ = _univ_mod._true_orig(
        con, date, shrcds=shrcds, exchcds=exchcds,
        min_price=min_price, min_mcap=min_mcap,
        attach_gvkey=attach_gvkey, attach_secid=attach_secid,
        attach_funda=attach_funda, require_options=require_options,
    )
    if attach_funda:
        missing = [c for c in _FUNDA_PATCH if c not in univ.columns]
        if missing and "gvkey" in univ.columns:
            asof = date if isinstance(date, _dt.date) else _dt.date.fromisoformat(str(date)[:10])
            pit  = (asof - _dt.timedelta(days=90)).isoformat()
            cols = ", ".join(["gvkey"] + missing)
            try:
                extra = con.execute(f"""
                    WITH ranked AS (
                        SELECT {cols},
                               ROW_NUMBER() OVER (
                                   PARTITION BY gvkey
                                   ORDER BY CAST(datadate AS DATE) DESC
                               ) AS _rn
                        FROM comp_funda
                        WHERE CAST(datadate AS DATE) <= DATE '{pit}'
                    )
                    SELECT {cols} FROM ranked WHERE _rn = 1
                """).pl()
                univ = univ.join(extra, on="gvkey", how="left")
            except Exception as exc:
                import warnings; warnings.warn(f"Funda patch failed for {missing}: {exc}")
    return univ

_univ_mod.get_universe_as_of = _patched_get_universe
print("Funda patch: installed on optlab.universe.get_universe_as_of")
print()
print("Both patches ready.")


In [ ]:
msf_info = con.execute(
    "SELECT MIN(CAST(date AS DATE)) AS min_date, MAX(CAST(date AS DATE)) AS max_date "
    "FROM crsp_msf"
).df()
display(msf_info)
CRSP_LAST = str(msf_info["max_date"].iloc[0])[:10]

BACKTEST_START = "2019-01-01"   # validation window; change to "1990-01-01" for full run
BACKTEST_END   = "2023-12-31"   # change to CRSP_LAST for full run
print(f"Backtest window: {BACKTEST_START}  ->  {BACKTEST_END}")


## 1. Signal validation: `gross_profitability`

This cell is the real smoke test for both patches.
If it prints "Rows:" and a count, both patches are working correctly.

**Expected:** 500â€“1 000 rows, ~20â€“35% nulls (financials have no COGS â€” correct).


In [ ]:
CHECK_DATE = "2023-06-30"
print(f"Computing gross_profitability as of {CHECK_DATE} ...")
try:
    sig_gp    = compute_signal("gross_profitability", CHECK_DATE, con, n_quantiles=5)
    null_rate = sig_gp["signal_value"].null_count() / sig_gp.height
    print(f"  Rows  : {sig_gp.height:,}")
    print(f"  Nulls : {sig_gp['signal_value'].null_count():,}  ({null_rate:.1%})")
    if null_rate > 0.35:
        print("  NOTE: >35% nulls expected â€” financials have no COGS.")
    print()
    print("Quintile counts:")
    display(sig_gp.group_by("signal_quantile").len().sort("signal_quantile").to_pandas())
    print()
    display(sig_gp["signal_value"].drop_nulls().describe().to_pandas())
except Exception as exc:
    print(f"FAILED: {exc}")
    raise


In [ ]:
vals = sig_gp["signal_value"].drop_nulls().to_numpy()
fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(vals, bins=80, color="seagreen", alpha=0.75, edgecolor="none")
ax.axvline(float(np.median(vals)), color="firebrick", lw=1.8, ls="--",
           label=f"Median: {np.median(vals):.3f}")
ax.axvline(0, color="black", lw=0.8, alpha=0.4)
ax.set_xlabel("GP = (Revenue âˆ’ COGS) / Total Assets", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_title(f"gross_profitability  --  {CHECK_DATE}", fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 2. Financial sector null-rate check

SIC 6xxx should be ~100% null. This confirms nulls are structural, not a data gap.


In [ ]:
try:
    _u = _patched_get_universe(con, _dt.date.fromisoformat(CHECK_DATE),
                                attach_gvkey=True, attach_funda=True)
    if "sich" in _u.columns:
        joined = (
            sig_gp.select(["permno","signal_value"])
            .join(_u.select(["permno","sich"])
                    .with_columns(pl.col("permno").cast(pl.Int64)),
                  on="permno", how="left")
            .with_columns((pl.col("sich") // 1000).cast(pl.Utf8).fill_null("?").alias("sic1"))
        )
        display(
            joined.group_by("sic1")
            .agg([pl.len().alias("n"),
                  pl.col("signal_value").is_null().sum().alias("n_null")])
            .with_columns((pl.col("n_null") / pl.col("n")).alias("null_rate"))
            .sort("sic1").to_pandas()
        )
    else:
        print("'sich' not in universe â€” skipping.")
except Exception as exc:
    print(f"Skipped: {exc}  (non-critical)")


## 3. Backtest: `gross_profitability`

Q5 (most profitable) long, Q1 (least profitable) short.
The patched `get_universe_as_of` is called automatically by the engine on each iteration.


In [ ]:
cfg_gp = BacktestConfig(
    signal="gross_profitability", start=BACKTEST_START, end=BACKTEST_END,
    universe="russell1000", portfolio="quintile_long_short",
    weighting="equal", n_quantiles=5, long_quantile=5, short_quantile=1,
)
print(f"Signal  : {cfg_gp.signal}  |  Long Q{cfg_gp.long_quantile}  Short Q{cfg_gp.short_quantile}")
print(f"Period  : {cfg_gp.start}  ->  {cfg_gp.end}")
t0 = time.time()
print("\nRunning ...")
result_gp = Backtest(cfg_gp).run(con)
print(f"Done in {(time.time()-t0)/60:.1f} min  ({result_gp.returns.height} months)")


In [ ]:
def _show_summary(result):
    s = result.summary()
    rows = [
        ("Period",                 f"{s['start']} -> {s['end']}"),
        ("Months",                 s["n_months"]),
        ("Avg N (long / short)",   f"{s['avg_n_long']:.0f} / {s['avg_n_short']:.0f}"),
        ("Ann. Return  L/S",       f"{s['ann_return_ls']:.2%}"        if s["ann_return_ls"]        else "N/A"),
        ("Ann. Vol     L/S",       f"{s['ann_vol_ls']:.2%}"           if s["ann_vol_ls"]           else "N/A"),
        ("Sharpe       L/S",       f"{s['sharpe_ls']:.3f}"            if s["sharpe_ls"]            else "N/A"),
        ("Max Drawdown L/S",       f"{s['max_drawdown_ls']:.2%}"      if s["max_drawdown_ls"]      else "N/A"),
        ("Monthly Win Rate",        f"{s['win_rate_ls']:.2%}"          if s["win_rate_ls"]          else "N/A"),
        ("Ann. Return  Long leg",  f"{s['ann_return_long']:.2%}"      if s["ann_return_long"]      else "N/A"),
        ("Ann. Return  Short leg", f"{s['ann_return_short']:.2%}"     if s["ann_return_short"]     else "N/A"),
        ("Avg Monthly Turnover",   f"{s['avg_monthly_turnover']:.2%}" if s["avg_monthly_turnover"] else "N/A"),
    ]
    display(pd.DataFrame(rows, columns=["Metric","Value"]).set_index("Metric"))

print("Gross profitability:")
_show_summary(result_gp)
result_gp.plot_cumulative(figsize=(13,5)); plt.show()
result_gp.plot_drawdown(figsize=(13,3));  plt.show()


In [ ]:
def _bm_table(result, bm):
    s, rows = result.summary(), []
    for k, info in bm.items():
        v = s.get(k); lo, hi = info.get("range",(None,None)); exp = info.get("expected")
        if v is not None and lo is not None:
            pct  = (v-exp)/abs(exp)*100 if exp else None
            flag = "PASS" if lo<=v<=hi else "CHECK"
        else:
            pct, flag = None, "N/A"
        rows.append({"Metric":info.get("label",k),
                     "Range" :f"[{lo:.3f},{hi:.3f}]" if lo is not None else "--",
                     "Actual":f"{v:.3f}" if v is not None else "N/A",
                     "Ref"   :f"{exp:.3f}" if exp else "--",
                     "% diff":f"{pct:+.1f}%" if pct is not None else "--",
                     "Result":flag})
    return pd.DataFrame(rows).set_index("Metric")

display(_bm_table(result_gp, {
    "ann_return_ls":{"label":"Ann Return L/S","expected":0.045,"range":(0.01,0.09),
                     "source":"Novy-Marx (2013)"},
    "sharpe_ls"    :{"label":"Sharpe L/S","expected":0.35,"range":(0.08,0.62)},
    "avg_monthly_turnover":{"label":"Avg Turnover","expected":0.12,"range":(0.05,0.25)},
}))


## 4. ROE comparison

In [ ]:
cfg_roe = BacktestConfig(
    signal="roe", start=BACKTEST_START, end=BACKTEST_END,
    universe="russell1000", portfolio="quintile_long_short",
    weighting="equal", n_quantiles=5, long_quantile=5, short_quantile=1,
)
print("Running ROE backtest ...")
t0 = time.time()
result_roe = Backtest(cfg_roe).run(con)
print(f"Done in {(time.time()-t0)/60:.1f} min")
print(); _show_summary(result_roe)


In [ ]:
rows = []
for name, res in [("gross_profitability", result_gp), ("roe", result_roe)]:
    s = res.summary()
    rows.append({"Signal":name,
                 "Ann Ret" :f"{s['ann_return_ls']:.2%}"        if s["ann_return_ls"]        else "N/A",
                 "Sharpe"  :f"{s['sharpe_ls']:.3f}"            if s["sharpe_ls"]            else "N/A",
                 "Max DD"  :f"{s['max_drawdown_ls']:.2%}"      if s["max_drawdown_ls"]      else "N/A",
                 "Turnover":f"{s['avg_monthly_turnover']:.2%}" if s["avg_monthly_turnover"] else "N/A"})
display(pd.DataFrame(rows).set_index("Signal"))

fig, ax = plt.subplots(figsize=(13,5))
for res, label, color in [(result_gp,"GP","seagreen"),(result_roe,"ROE","dodgerblue")]:
    s = res.returns.to_pandas().set_index("date")["ls_ret"].fillna(0)
    ax.plot((1+s).cumprod()-1, label=label, color=color, lw=1.8)
common = pd.to_datetime(sorted(
    set(result_gp.returns.to_pandas()["date"]) &
    set(result_roe.returns.to_pandas()["date"])
))
gps  = result_gp.returns.to_pandas().assign(date=lambda d:pd.to_datetime(d["date"])).set_index("date").reindex(common)["ls_ret"].fillna(0)
roes = result_roe.returns.to_pandas().assign(date=lambda d:pd.to_datetime(d["date"])).set_index("date").reindex(common)["ls_ret"].fillna(0)
ax.plot((1+(gps+roes)/2).cumprod()-1, label="50/50 mix", color="darkorange", lw=2.0, ls="--")
ax.axhline(0, color="black", lw=0.8, ls="--", alpha=0.4)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_title("Quality factor variants â€” cumulative L/S returns", fontsize=11)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


## 5. Novy-Marx correlation: GP vs B/M

Expected Spearman rank correlation: **âˆ’0.15 to âˆ’0.35**.
Negative = GP and value are cross-sectional complements (Novy-Marx 2013 core result).


In [ ]:
from scipy.stats import spearmanr
try:
    sig_bm = compute_signal("book_to_market", CHECK_DATE, con, n_quantiles=5)
    both   = (
        sig_gp.select(["permno","signal_rank"]).rename({"signal_rank":"gp_rank"})
        .join(sig_bm.select(["permno","signal_rank"]).rename({"signal_rank":"bm_rank"}),
              on="permno", how="inner").drop_nulls()
    )
    rho, pval = spearmanr(both["gp_rank"].to_numpy(), both["bm_rank"].to_numpy())
    print(f"GP vs B/M  rho={rho:.3f}  p={pval:.4f}  n={both.height:,}")
    print({"PASS":"PASS: complements confirmed.",
           "BORDERLINE":"BORDERLINE: near-zero â€” check definitions.",
           "WARNING":"WARNING: positive â€” investigate."}[
          "PASS" if rho<-0.05 else ("BORDERLINE" if rho<0.05 else "WARNING")])

    fig, ax = plt.subplots(figsize=(6,5))
    ax.scatter(both["bm_rank"].to_numpy(), both["gp_rank"].to_numpy(),
               alpha=0.15, s=6, color="teal")
    ax.set_xlabel("B/M rank"); ax.set_ylabel("GP rank")
    ax.set_title(f"GP vs B/M  --  {CHECK_DATE}  (rho={rho:.3f})", fontsize=11)
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
except Exception as exc:
    print(f"Skipped: {exc}  (non-critical)")


## 6. Permanent fixes (when you have time)

| Issue | Notebook fix | Permanent fix in optlab source |
|---|---|---|
| `optcrsp_link` missing | Empty stub view | `optlab/universe.py`: make the `LEFT JOIN optcrsp_link` conditional on `attach_secid=True` |
| `cogs` missing from `attach_funda` | Monkey-patch injects it | `optlab/universe.py`: add `cogs` to the Compustat `SELECT` list |

**Week 5:** `tcost_bps=10` â€” quality has low turnover; cost impact small.  
**Week 6:** GP should load ~0.5â€“1.0 on RMW in FF6 attribution.


In [ ]:
for name, res, path in [
    ("gross_profitability", result_gp,  "outputs/03_quality_gross_profitability"),
    ("roe",                 result_roe, "outputs/03_quality_roe"),
]:
    res.save(path); print(f"Saved: {path}/")
